
- conda install git
- pip install pillow (then restart kernel)
- pip install evaluate scikit-learn nltk rouge-score

In [ ]:
# !pip install -q git+https://github.com/huggingface/transformers.git
# !pip install -q accelerate datasets peft bitsandbytes
# !pip install pillow
# # then restart kernel

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

In [ ]:
from datasets import load_dataset
ds = load_dataset("SimulaMet-HOST/Kvasir-VQA")["raw"]
ds

/home/ebmi/anaconda3/envs/kvasir-vlm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['image', 'source', 'question', 'answer', 'img_id'],
    num_rows: 58849
})

In [ ]:
import os
from datasets import Dataset, Features, Image, Value, load_dataset, DatasetDict
from PIL import Image as PILImage
import random
import pandas as pd
from collections import defaultdict

/home/ebmi/anaconda3/envs/kvasir-vlm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from datasets import concatenate_datasets

valid_ds = ds.filter(lambda ex: ex["question"] and ex["question"] != "none")

abnormal_ids = set(
    ex["img_id"] for ex in valid_ds if ex["source"].lower() != "normal"
)

seen_ids = set()
added_examples = []
for ex in valid_ds:
    if ex["img_id"] in abnormal_ids and ex["img_id"] not in seen_ids:
        added_examples.append({
            "image": ex["image"],
            "source": ex["source"],
            "question": "Does this image contain any finding?",
            "answer": "yes",
            "img_id": ex["img_id"]
        })
        seen_ids.add(ex["img_id"])

modified_ds = Dataset.from_list(added_examples)
cleaned_ds = concatenate_datasets([valid_ds, modified_ds])

In [ ]:
print("Original size:", len(ds))
print("Cleaned size: ", len(valid_ds))
print("final modified data size: ", len(cleaned_ds))
print("new added QAs: ", len(added_examples))

Original size: 58849
Cleaned size:  58798
final modified data size:  62747
new added QAs:  3949


Split 80/10/10

In [ ]:
import random
from datasets import load_dataset, DatasetDict

all_ids = sorted(set(cleaned_ds["img_id"]))
random.seed(42)
random.shuffle(all_ids)

n = len(all_ids)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
train_ids = set(all_ids[:n_train])
val_ids   = set(all_ids[n_train : n_train + n_val])
test_ids  = set(all_ids[n_train + n_val :])

In [ ]:
def filter_by_id(example, id_set):
    return example["img_id"] in id_set

In [ ]:
train_ds = cleaned_ds.filter(lambda ex: filter_by_id(ex, train_ids), batched=False)
val_ds   = cleaned_ds.filter(lambda ex: filter_by_id(ex, val_ids),   batched=False)
test_ds  = cleaned_ds.filter(lambda ex: filter_by_id(ex, test_ids),  batched=False)

In [ ]:
dataset = DatasetDict({
    "train":      train_ds,
    "validation": val_ds,
    "test":       test_ds,
})
print({k: len(v) for k, v in dataset.items()})

{'train': 50105, 'validation': 6287, 'test': 6355}


In [ ]:
dataset_full = dataset
data = dataset_full.remove_columns(["source", "img_id"])
data

DatasetDict({
    train: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 50105
    })
    validation: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6287
    })
    test: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6355
    })
})

No (image, Q, A) rows has a flipped/rotated image paired with a position related question, and no color transform is applied, so no label can become invalid.
if the question is not position related, then geometric transformation is not a problem. img can be of that type too. we just don't have ground truth if we apply geometric transformation to image with position related question.


In [ ]:
from datasets import DatasetDict
from PIL import Image as PILImage


def t_identity(img):  return img
def t_hflip(img):     return img.transpose(PILImage.FLIP_LEFT_RIGHT)
def t_vflip(img):     return img.transpose(PILImage.FLIP_TOP_BOTTOM)
def t_rot90(img):     return img.transpose(PILImage.ROTATE_90)
def t_rot180(img):    return img.transpose(PILImage.ROTATE_180)
def t_rot270(img):    return img.transpose(PILImage.ROTATE_270)
def t_transpose(img): return img.transpose(PILImage.TRANSPOSE)
def t_transverse(img):return img.transpose(PILImage.TRANSVERSE)

def _rotate_keep(img, angle):

    rotated = img.rotate(angle, resample=PILImage.BICUBIC, expand=True)
    return rotated.resize(img.size, PILImage.BICUBIC)

def t_rot_p12(img):   return _rotate_keep(img, 12)
def t_rot_m12(img):   return _rotate_keep(img, -12)

# for non-position related QA instances
GEO_TRANSFORMS = [t_hflip, t_vflip, t_rot90, t_rot180, t_rot270,
                  t_transpose, t_transverse, t_rot_p12, t_rot_m12]
assert len(GEO_TRANSFORMS) == 9

N_COPIES = 10  # total variants per original row ( 10x)

# Position questions: the three "Where in the image is ...?" questions.
def is_position_question(q):
    return "where in the image" in q.lower()

In [ ]:
def augment_batch(batch):
    imgs = batch["image"]
    qs   = batch["question"]
    ans  = batch["answer"]

    new_imgs, new_qs, new_ans = [], [], []

    for img, q0, a in zip(imgs, qs, ans):
        img = img.convert("RGB")

        if is_position_question(q0):
            # if POSITION related question, then identity copies only (no geometry). 
            for _ in range(N_COPIES):
                new_imgs.append(img); new_qs.append(q0); new_ans.append(a)
        else:
            # NON-position question -> original + 9 geometric variants.
            new_imgs.append(img); new_qs.append(q0); new_ans.append(a)  # original
            for fn in GEO_TRANSFORMS:
                new_imgs.append(fn(img)); new_qs.append(q0); new_ans.append(a)

    return {"image": new_imgs, "question": new_qs, "answer": new_ans}

In [ ]:
import os
from datasets import DatasetDict, load_from_disk

SAVE_PATH = "kvasir_strict_clean10x"   # folder on disk

print("old train size:", len(data["train"]))

new_train = data["train"].map(
    augment_batch,
    batched=True,
    batch_size=16,          
    writer_batch_size=100,  
    remove_columns=data["train"].column_names,
    load_from_cache_file=False,
    desc="strict-clean 10x…"
)

data = DatasetDict({
    "train":      new_train,
    "validation": data["validation"],
    "test":       data["test"],
})

print("new train size:", len(data["train"]))  


data.save_to_disk(SAVE_PATH)
print("saved to:", os.path.abspath(SAVE_PATH))

In [ ]:
import os
from datasets import load_from_disk

SAVE_PATH ="/data/shanto/kvasir-vqa//kvasir_strict_clean10x" 


if os.path.exists(SAVE_PATH):
    data = load_from_disk(SAVE_PATH)
    print("reloaded:", {k: len(v) for k, v in data.items()})
else:
    print("No saved dataset found — run the augmentation cell first.")

reloaded: {'train': 501050, 'validation': 6287, 'test': 6355}


## load form saved clean10x

In [ ]:

import os
from datasets import load_from_disk

SAVE_PATH = "/data/shanto/kvasir-vqa/kvasir_strict_clean10x"

if os.path.exists(SAVE_PATH):
    data = load_from_disk(SAVE_PATH)
    print("reloaded:", {k: len(v) for k, v in data.items()})
else:
    raise FileNotFoundError(f"No saved dataset at {SAVE_PATH} — build/copy it first.")

/home/ebmi/anaconda3/envs/kvasir-vlm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


reloaded: {'train': 501050, 'validation': 6287, 'test': 6355}


In [ ]:
print("train  :", len(data["train"]))
print("val    :", len(data["validation"]))
print("test   :", len(data["test"]))

train  : 501050
val    : 6287
test   : 6355


In [ ]:
data

DatasetDict({
    train: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 501050
    })
    validation: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6287
    })
    test: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6355
    })
})

In [ ]:
import torch
from peft import LoraConfig
from transformers import BitsAndBytesConfig, PaliGemmaForConditionalGeneration, PaliGemmaProcessor

DEVICE = torch.device("cuda:0")
USE_LORA = True
USE_8BIT = True


print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())
print("Using device:", DEVICE)
print("GPU name:", torch.cuda.get_device_name(0))


processor = PaliGemmaProcessor.from_pretrained(
    "google/paligemma2-3b-pt-224",
    do_image_splitting=False,
    use_fast=True
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    use_dora=True,
    init_lora_weights="gaussian"
)

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)

model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma2-3b-pt-224",
    quantization_config=bnb_config,
    device_map={"": 0}
    # device_map="auto",
)

model.add_adapter(lora_config)
model.enable_adapters()

In [ ]:
def print_trainable_parameters(model):
    trainable = total = 0
    for _, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad: trainable += p.numel()
    print(f"Trainable parameters: {trainable:,}")
    print(f"Total parameters: {total:,}")
    print(f"Percentage of trainable parameters: {100*trainable/total:.2f}%")

print_trainable_parameters(model)

In [ ]:
class PaligemmaVQACollator:
    def __init__(self, processor, device):
        self.processor = processor
        self.device = device
    def __call__(self, examples):
        prompts = [f"<image> Answer as a medical specialist. {ex['question']}" for ex in examples]
        suffixes = [ex["answer"] for ex in examples]
        images = [ex["image"].convert("RGB") for ex in examples]
        batch = self.processor(text=prompts, images=images, suffix=suffixes,
                               return_tensors="pt", padding="longest")
        batch = {k: v.to(self.device) for k, v in batch.items()}
        return batch

data_collator = PaligemmaVQACollator(processor, DEVICE)

In [ ]:
batch_sample = [data['train'][0], data['train'][1]]
batch_inputs = data_collator(batch_sample)
for k, v in batch_inputs.items():
    print(f"{k}: {v.shape}")

input_ids: torch.Size([2, 283])
token_type_ids: torch.Size([2, 283])
attention_mask: torch.Size([2, 283])
pixel_values: torch.Size([2, 3, 224, 224])
labels: torch.Size([2, 283])


## Training



In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=50,
    learning_rate=1e-4,
    weight_decay=0.01,
    logging_steps=25,
    output_dir=r"paligemma3bpt224_lora_clean10x_checkpoint",
    save_strategy="epoch",
    save_steps=250,
    save_total_limit=1,
    eval_strategy="epoch",
    bf16=True,
    remove_unused_columns=False,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=data['train'],
    eval_dataset=data['validation'],
)

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
trainer.train()

## Evaluation on clean test set

In [ ]:


import os, gc, torch
from peft import PeftModel
from transformers import PaliGemmaForConditionalGeneration

CKPT = "paligemma3bpt224_lora_clean10x_checkpoint/checkpoint-93948"


print("checkpoint contents:", os.listdir(CKPT))

try:
    del model
    gc.collect(); torch.cuda.empty_cache()
except NameError:
    pass


base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma2-3b-pt-224",
    quantization_config=bnb_config,  
    device_map={"": 0},
)


model = PeftModel.from_pretrained(base_model, CKPT, is_trainable=False)
model.eval()

print("loaded adapter from:", CKPT)
print("device:", next(model.parameters()).device)

In [ ]:
import evaluate
import torch
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import Levenshtein

def clean_answer(text):
    lines = text.strip().split("\n")
    return lines[-1].strip() if lines else text.strip()

In [ ]:
preds, refs = [], []
model.eval()

for ex in tqdm(data['test']):
    image = ex["image"].convert("RGB")
    question = ex["question"]
    answer = ex["answer"]
    prompt = f"<image> Answer as a medical specialist. {question}"
    inputs = processor(text=prompt, images=image, return_tensors="pt",
                       padding="longest").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=50, num_beams=1,
                                     do_sample=False, use_cache=False)
    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    preds.append(clean_answer(pred))
    refs.append(answer if isinstance(answer, list) else [answer])

  0%|          | 0/6355 [00:00<?, ?it/s][transformers] The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/home/ebmi/anaconda3/envs/kvasir-vlm/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
100%|██████████| 6355/6355 [2:31:09<00:00,  1.43s/it]  


In [ ]:
refs_single = [r[0] for r in refs]
accuracy = sum(p in r for p, r in zip(preds, refs)) / len(refs) * 100

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

bleu_res   = bleu.compute(predictions=preds, references=[[r] for r in refs_single])
rouge_res  = rouge.compute(predictions=preds, references=refs_single)
meteor_res = meteor.compute(predictions=preds, references=refs_single)

j_scores = []
for r, p in zip(refs_single, preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard = sum(j_scores) / len(j_scores) * 100

vectorizer = TfidfVectorizer().fit(refs_single + preds)
ref_vecs  = vectorizer.transform(refs_single)
pred_vecs = vectorizer.transform(preds)
cosine = cosine_similarity(ref_vecs, pred_vecs).diagonal().mean() * 100

def normalized_levenshtein(s1, s2):
    if not s1 and not s2: return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def average_levenshtein_similarity(refs, preds):
    sims = []
    for r, p in zip(refs, preds):
        ref = r[0] if isinstance(r, list) else r
        sims.append(1 - normalized_levenshtein(ref, p))
    return sum(sims) / len(sims) * 100

levenshtein_score = average_levenshtein_similarity(refs, preds)

In [ ]:
results = {
    "accuracy (%)": round(accuracy, 2),
    "bleu": bleu_res,
    "rouge": rouge_res,
    "meteor": meteor_res,
    "jaccard (%)": round(jaccard, 2),
    "cosine (%)": round(cosine, 2),
    "levenshtein (%)": round(levenshtein_score, 2)
}
for k, v in results.items():
    print(f"{k}:\n{v}\n")

accuracy (%):
87.71

bleu:
{'bleu': 0.8537199393977739, 'precisions': [0.9076421036654777, 0.8591322192905306, 0.8281600598354525, 0.8225698324022347], 'brevity_penalty': 1.0, 'length_ratio': 1.006877053564606, 'translation_length': 13177, 'reference_length': 13087}

rouge:
{'rouge1': np.float64(0.9253879681245334), 'rouge2': np.float64(0.18621281162030778), 'rougeL': np.float64(0.9237733159255291), 'rougeLsum': np.float64(0.9239366026219797)}

meteor:
{'meteor': np.float64(0.538848654577934)}

jaccard (%):
90.5

cosine (%):
77.76

levenshtein (%):
92.36



## Adversarial Robustness

In [ ]:
synonyms_dict = {
    "type": ["kind", "category"],
    "procedure": ["process", "test"],
    "image": ["picture", "visual"],
    "abnormalities": ["irregularities", "issues"],
    "present": ["visible", "detected"],
    "easy": ["simple", "straightforward"],
    "detect": ["identify", "locate"],
    "polyp": ["lesion", "mass", "growth"],
    "size": ["dimension", "measurement"],
    "instrument": ["tool", "equipment"],
    "removed": ["extracted", "taken out"],
    "where": ["in what part", "in which area", "location of"],
    "how many": ["number of", "count of", "total"]
}

insert_words = [
    "possibly", "likely", "evidently", "apparently", "visibly",
    "clinically", "endoscopically", "approximately", "typically"
]
import random

def synonym_replacement(question, synonyms_dict):
    words = question.split()
    new_words = [random.choice(synonyms_dict.get(w.lower(), [w])) for w in words]
    return " ".join(new_words)

def random_insertion(question, insert_words):
    words = question.split()
    if not words: return question
    insert_word = random.choice(insert_words)
    insert_pos = random.randint(0, len(words))
    return " ".join(words[:insert_pos] + [insert_word] + words[insert_pos:])

def random_deletion(question, p=0.2):
    words = question.split()
    if len(words) == 1: return question
    return " ".join([w for w in words if random.random() > p])


In [ ]:
attacked_preds = []

model.eval()

for ex in tqdm(data['test']):
    image = ex["image"].convert("RGB")
    question = ex["question"]
    answer = ex["answer"]

    # Apply only synonym replacement
    attacked_q = synonym_replacement(question, synonyms_dict)

    prompt = f"<image> Answer as a medical specialist. {attacked_q}"

    # Tokenize with image and prompt
    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt",
        padding="longest"
    ).to(model.device)

    # Generate prediction
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            use_cache=False
        )

    # Decode and clean prediction
    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)

    attacked_preds.append(pred)

  0%|          | 0/6355 [00:00<?, ?it/s]/home/ebmi/anaconda3/envs/kvasir-vlm/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
100%|██████████| 6355/6355 [2:31:49<00:00,  1.43s/it]  


In [ ]:
# Acc@Attack: Percentage of correct predictions under attack
acc_at_attack = sum(p in r for p, r in zip(attacked_preds, refs)) / len(refs) * 100

# ASR: Attack success rate (when prediction changes)
asr = sum(p1 != p2 for p1, p2 in zip(preds, attacked_preds)) / len(preds) * 100

print(f"Acc@Attack (%): {acc_at_attack:.3f}")
print(f"Attack Success Rate (ASR %): {asr:.3f}")


Acc@Attack (%): 83.950
Attack Success Rate (ASR %): 6.593


In [ ]:
# Prepare flat reference list
refs_single = [r[0] for r in refs]

# Accuracy under adversarial attack
adv_accuracy = sum(p in r for p, r in zip(attacked_preds, refs)) / len(refs) * 100

# Evaluate metrics
bleu_res_adv   = bleu.compute(predictions=attacked_preds, references=[[r] for r in refs_single])
rouge_res_adv  = rouge.compute(predictions=attacked_preds, references=refs_single)
meteor_res_adv = meteor.compute(predictions=attacked_preds, references=refs_single)

# Jaccard Similarity
j_scores_adv = []
for r, p in zip(refs_single, attacked_preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores_adv.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard_adv = sum(j_scores_adv) / len(j_scores_adv) * 100

# Cosine Similarity (TF-IDF)
vectorizer_adv = TfidfVectorizer().fit(refs_single + attacked_preds)
ref_vecs_adv  = vectorizer_adv.transform(refs_single)
pred_vecs_adv = vectorizer_adv.transform(attacked_preds)
cos_sims_adv  = cosine_similarity(ref_vecs_adv, pred_vecs_adv).diagonal()
cosine_adv = cos_sims_adv.mean() * 100

# Levenshtein Similarity
def normalized_levenshtein(s1, s2):
    if not s1 and not s2:
        return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def similarity_score(a_ij, o_q_i, tau=0.5):
    nl = normalized_levenshtein(a_ij, o_q_i)
    return 1 - nl if nl < tau else 0

def average_levenshtein_similarity(ground_truth, predicted):
    total_score = 0
    for refs, pred in zip(ground_truth, predicted):
        if not pred:
            continue
        max_score = max(similarity_score(ref, pred) for ref in refs)
        total_score += max_score
    return total_score / len(ground_truth) * 100

levenshtein_adv = average_levenshtein_similarity(refs, attacked_preds)

# Collect and print results
adv_results = {
    "accuracy (%)": round(adv_accuracy, 2),
    "bleu": bleu_res_adv,
    "rouge": rouge_res_adv,
    "meteor": meteor_res_adv,
    "jaccard (%)": round(jaccard_adv, 2),
    "cosine (%)": round(cosine_adv, 2),
    "levenshtein (%)": round(levenshtein_adv, 2)
}

for k, v in adv_results.items():
    print(f"{k}:\n{v}\n")


accuracy (%):
83.95

bleu:
{'bleu': 0.8011398954101612, 'precisions': [0.8691814409656733, 0.8068115942028985, 0.770153730783652, 0.7627345844504021], 'brevity_penalty': 1.0, 'length_ratio': 1.0128371666539313, 'translation_length': 13255, 'reference_length': 13087}

rouge:
{'rouge1': np.float64(0.8933649207358919), 'rouge2': np.float64(0.1765439247379748), 'rougeL': np.float64(0.8918656392988309), 'rougeLsum': np.float64(0.8919485081147089)}

meteor:
{'meteor': np.float64(0.5180977264226364)}

jaccard (%):
86.98

cosine (%):
74.73

levenshtein (%):
87.89



## with all three types of perturbations

In [ ]:
attacked_preds = []

model.eval()

for ex in tqdm(data['test']):
    image = ex["image"].convert("RGB")
    question = ex["question"]
    answer = ex["answer"]

    #applying all text perturbations
    attacked_q = synonym_replacement(question, synonyms_dict)
    attacked_q = random_insertion(attacked_q, insert_words)
    attacked_q = random_deletion(attacked_q)

  
    prompt = f"<image> Answer as a medical specialist. {attacked_q}"

    # Tokenize with image and prompt
    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt",
        padding="longest"
    ).to(model.device)

    # Generate prediction
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            use_cache=False
        )

    # Decode and clean prediction
    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)

    attacked_preds.append(pred)


  0%|          | 0/6355 [00:00<?, ?it/s]/home/ebmi/anaconda3/envs/kvasir-vlm/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
100%|██████████| 6355/6355 [2:22:36<00:00,  1.35s/it]  


In [ ]:
# Acc@Attack: Percentage of correct predictions under attack
acc_at_attack = sum(p in r for p, r in zip(attacked_preds, refs)) / len(refs) * 100

# ASR: Attack success rate (when prediction changes)
asr = sum(p1 != p2 for p1, p2 in zip(preds, attacked_preds)) / len(preds) * 100

print(f"Acc@Attack (%): {acc_at_attack:.2f}")
print(f"Attack Success Rate (ASR %): {asr:.2f}")

Acc@Attack (%): 72.21
Attack Success Rate (ASR %): 20.82


In [ ]:
# Prepare flat reference list
refs_single = [r[0] for r in refs]

# Accuracy under adversarial attack
adv_accuracy = sum(p in r for p, r in zip(attacked_preds, refs)) / len(refs) * 100

# Evaluate metrics
bleu_res_adv   = bleu.compute(predictions=attacked_preds, references=[[r] for r in refs_single])
rouge_res_adv  = rouge.compute(predictions=attacked_preds, references=refs_single)
meteor_res_adv = meteor.compute(predictions=attacked_preds, references=refs_single)

# Jaccard Similarity
j_scores_adv = []
for r, p in zip(refs_single, attacked_preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores_adv.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard_adv = sum(j_scores_adv) / len(j_scores_adv) * 100

# Cosine Similarity (TF-IDF)
vectorizer_adv = TfidfVectorizer().fit(refs_single + attacked_preds)
ref_vecs_adv  = vectorizer_adv.transform(refs_single)
pred_vecs_adv = vectorizer_adv.transform(attacked_preds)
cos_sims_adv  = cosine_similarity(ref_vecs_adv, pred_vecs_adv).diagonal()
cosine_adv = cos_sims_adv.mean() * 100

# Levenshtein Similarity
def normalized_levenshtein(s1, s2):
    if not s1 and not s2:
        return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def similarity_score(a_ij, o_q_i, tau=0.5):
    nl = normalized_levenshtein(a_ij, o_q_i)
    return 1 - nl if nl < tau else 0

def average_levenshtein_similarity(ground_truth, predicted):
    total_score = 0
    for refs, pred in zip(ground_truth, predicted):
        if not pred:
            continue
        max_score = max(similarity_score(ref, pred) for ref in refs)
        total_score += max_score
    return total_score / len(ground_truth) * 100

levenshtein_adv = average_levenshtein_similarity(refs, attacked_preds)

# Collect and print results
adv_results = {
    "accuracy (%)": round(adv_accuracy, 2),
    "bleu": bleu_res_adv,
    "rouge": rouge_res_adv,
    "meteor": meteor_res_adv,
    "jaccard (%)": round(jaccard_adv, 2),
    "cosine (%)": round(cosine_adv, 2),
    "levenshtein (%)": round(levenshtein_adv, 2)
}

for k, v in adv_results.items():
    print(f"{k}:\n{v}\n")


accuracy (%):
72.21

bleu:
{'bleu': 0.6678289024089247, 'precisions': [0.7501183525327442, 0.6869757873081184, 0.6609776654024442, 0.6652903225806451], 'brevity_penalty': 0.9679388188707604, 'length_ratio': 0.9684419653090853, 'translation_length': 12674, 'reference_length': 13087}

rouge:
{'rouge1': np.float64(0.7665648106411603), 'rouge2': np.float64(0.14679658407586543), 'rougeL': np.float64(0.7651220589961282), 'rougeLsum': np.float64(0.7649796636588388)}

meteor:
{'meteor': np.float64(0.4420248627289099)}

jaccard (%):
74.65

cosine (%):
65.64

levenshtein (%):
75.35



## Image Perturbations

In [ ]:
from PIL import Image, ImageFilter, ImageEnhance
import torchvision.transforms as T
import numpy as np
import io

# Gaussian noise
def add_gaussian_noise(img, mean=0, std=10):
    np_img = np.array(img).astype(np.float32)
    noise = np.random.normal(mean, std, np_img.shape)
    noisy_img = np.clip(np_img + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_img)

# Gaussian blur
def apply_blur(img, radius=2):
    return img.filter(ImageFilter.GaussianBlur(radius))

# Brightness shift
def shift_brightness(img, factor=1.5):  # >1 brightens, <1 darkens
    enhancer = ImageEnhance.Brightness(img)
    return enhancer.enhance(factor)

# JPEG compression
def jpeg_compress(img, quality=30):
    buffer = io.BytesIO()
    img.save(buffer, format='JPEG', quality=quality)
    return Image.open(buffer)


In [ ]:
image_attacked_preds = []

model.eval()

for ex in tqdm(data['test']):
    image = ex["image"].convert("RGB")  # Ensure RGB
    question = ex["question"]
    answer = ex["answer"]

    # Apply image perturbations
    perturbed_image = add_gaussian_noise(image)
    perturbed_image = apply_blur(perturbed_image)
    perturbed_image = shift_brightness(perturbed_image, factor=0.7)
    # perturbed_image = jpeg_compress(image, quality=25)

    # Build prompt without chat template
    prompt = f"<image> Answer as a medical specialist. {question}"

    # Prepare input
    inputs = processor(
        text=prompt,
        images=perturbed_image,
        return_tensors="pt",
        padding="longest"
    ).to(model.device)

    # Generate prediction
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            use_cache=False
        )

    # Decode and clean
    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)
    image_attacked_preds.append(pred)


100%|██████████| 6355/6355 [2:34:57<00:00,  1.46s/it]  


In [ ]:
# Accuracy under image perturbation
acc_at_attack_img = sum(p in r for p, r in zip(image_attacked_preds, refs)) / len(refs) * 100

# Attack success rate
asr_img = sum(p1 != p2 for p1, p2 in zip(preds, image_attacked_preds)) / len(preds) * 100

print(f"Image Acc@Attack (%): {acc_at_attack_img:.2f}")
print(f"Image ASR (%): {asr_img:.2f}")


Image Acc@Attack (%): 87.41
Image ASR (%): 6.42


In [ ]:
# Flatten reference list
refs_single = [r[0] for r in refs]

# Accuracy under attack
adv_accuracy = sum(p in r for p, r in zip(image_attacked_preds, refs)) / len(refs) * 100

# Metrics
bleu_res_adv   = bleu.compute(predictions=image_attacked_preds, references=[[r] for r in refs_single])
rouge_res_adv  = rouge.compute(predictions=image_attacked_preds, references=refs_single)
meteor_res_adv = meteor.compute(predictions=image_attacked_preds, references=refs_single)

# Jaccard Similarity
j_scores_adv = []
for r, p in zip(refs_single, image_attacked_preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores_adv.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard_adv = sum(j_scores_adv) / len(j_scores_adv) * 100

# Cosine Similarity (TF-IDF)
vectorizer_adv = TfidfVectorizer().fit(refs_single + image_attacked_preds)
ref_vecs_adv  = vectorizer_adv.transform(refs_single)
pred_vecs_adv = vectorizer_adv.transform(image_attacked_preds)
cos_sims_adv  = cosine_similarity(ref_vecs_adv, pred_vecs_adv).diagonal()
cosine_adv = cos_sims_adv.mean() * 100

# Levenshtein Similarity
levenshtein_adv = average_levenshtein_similarity(refs, image_attacked_preds)

# Pack results
adv_results = {
    "accuracy (%)": round(adv_accuracy, 2),
    "bleu": bleu_res_adv,
    "rouge": rouge_res_adv,
    "meteor": meteor_res_adv,
    "jaccard (%)": round(jaccard_adv, 2),
    "cosine (%)": round(cosine_adv, 2),
    "levenshtein (%)": round(levenshtein_adv, 2)
}

# Display
for k, v in adv_results.items():
    print(f"{k}:\n{v}\n")


accuracy (%):
87.41

bleu:
{'bleu': 0.8520323339795739, 'precisions': [0.9039689781021898, 0.8534647638664117, 0.8269122345423793, 0.8260869565217391], 'brevity_penalty': 1.0, 'length_ratio': 1.004966760907771, 'translation_length': 13152, 'reference_length': 13087}

rouge:
{'rouge1': np.float64(0.9223061990179066), 'rouge2': np.float64(0.18525439381185188), 'rougeL': np.float64(0.9207798576916402), 'rougeLsum': np.float64(0.9208919447221714)}

meteor:
{'meteor': np.float64(0.5363344480037041)}

jaccard (%):
90.09

cosine (%):
77.35

levenshtein (%):
90.86



### lowering image quality

In [ ]:
image_attacked_preds = []

model.eval()

for ex in tqdm(data['test']):
    image = ex["image"].convert("RGB")  # Ensure RGB
    question = ex["question"]
    answer = ex["answer"]

    # Apply image perturbations

    perturbed_image = jpeg_compress(image, quality=25)

    # Build prompt without chat template
    prompt = f"<image> Answer as a medical specialist. {question}"

    # Prepare input
    inputs = processor(
        text=prompt,
        images=perturbed_image,
        return_tensors="pt",
        padding="longest"
    ).to(model.device)

    # Generate prediction
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            use_cache=False
        )

    # Decode and clean
    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)
    image_attacked_preds.append(pred)


100%|██████████| 6355/6355 [2:31:41<00:00,  1.43s/it]  


In [ ]:
# Accuracy under image perturbation
acc_at_attack_img = sum(p in r for p, r in zip(image_attacked_preds, refs)) / len(refs) * 100

# Attack success rate
asr_img = sum(p1 != p2 for p1, p2 in zip(preds, image_attacked_preds)) / len(preds) * 100

print(f"Image Acc@Attack (%): {acc_at_attack_img:.2f}")
print(f"Image ASR (%): {asr_img:.2f}")

Image Acc@Attack (%): 87.60
Image ASR (%): 3.56


In [ ]:
# Flatten reference list
refs_single = [r[0] for r in refs]

# Accuracy under attack
adv_accuracy = sum(p in r for p, r in zip(image_attacked_preds, refs)) / len(refs) * 100

# Metrics
bleu_res_adv   = bleu.compute(predictions=image_attacked_preds, references=[[r] for r in refs_single])
rouge_res_adv  = rouge.compute(predictions=image_attacked_preds, references=refs_single)
meteor_res_adv = meteor.compute(predictions=image_attacked_preds, references=refs_single)

# Jaccard Similarity
j_scores_adv = []
for r, p in zip(refs_single, image_attacked_preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores_adv.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard_adv = sum(j_scores_adv) / len(j_scores_adv) * 100

# Cosine Similarity (TF-IDF)
vectorizer_adv = TfidfVectorizer().fit(refs_single + image_attacked_preds)
ref_vecs_adv  = vectorizer_adv.transform(refs_single)
pred_vecs_adv = vectorizer_adv.transform(image_attacked_preds)
cos_sims_adv  = cosine_similarity(ref_vecs_adv, pred_vecs_adv).diagonal()
cosine_adv = cos_sims_adv.mean() * 100

# Levenshtein Similarity
levenshtein_adv = average_levenshtein_similarity(refs, image_attacked_preds)

# Pack results
adv_results = {
    "accuracy (%)": round(adv_accuracy, 2),
    "bleu": bleu_res_adv,
    "rouge": rouge_res_adv,
    "meteor": meteor_res_adv,
    "jaccard (%)": round(jaccard_adv, 2),
    "cosine (%)": round(cosine_adv, 2),
    "levenshtein (%)": round(levenshtein_adv, 2)
}

# Display
for k, v in adv_results.items():
    print(f"{k}:\n{v}\n")


accuracy (%):
87.6

bleu:
{'bleu': 0.8538218771637313, 'precisions': [0.907151533556028, 0.8585888220624909, 0.8292774241856983, 0.8228187919463087], 'brevity_penalty': 1.0, 'length_ratio': 1.0064949950332391, 'translation_length': 13172, 'reference_length': 13087}

rouge:
{'rouge1': np.float64(0.9240984884495106), 'rouge2': np.float64(0.18674687403522172), 'rougeL': np.float64(0.922296789846786), 'rougeLsum': np.float64(0.9223973076241581)}

meteor:
{'meteor': np.float64(0.5377895321538269)}

jaccard (%):
90.32

cosine (%):
77.63

levenshtein (%):
91.2

